In [ ]:
#!/usr/bin/env python3
"""
Script to analyze inference results
Used to compute distributions of harmful_request and harmful_response
"""

import json
import os
import argparse
from collections import Counter, defaultdict
from pathlib import Path


def load_results_from_directory(results_dir):
    """Load all JSON result files from a directory"""
    results = []
    json_files = list(Path(results_dir).glob("result_*.json"))
    
    print(f"Found {len(json_files)} result files")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                results.append(data)
        except Exception as e:
            print(f"Failed to load file {json_file}: {e}")
    
    return results


def analyze_harmful_requests(results):
    """Analyze distribution of harmful_request"""
    harmful_requests = []
    for result in results:
        parsed_response = result.get('parsed_response', {})
        harmful_request = parsed_response.get('harmful_request', 'unknown')
        harmful_requests.append(harmful_request)
    
    harmful_request_counts = Counter(harmful_requests)
    return harmful_request_counts


def analyze_harmful_responses(results, task):
    """Analyze distribution of harmful_response"""
    harmful_responses = []
    # Load exclusion lists
    exclude_ids_by_task = {
        "vague": ["3I02618YA14S7SHVOPAYBZTIDARUPP", "3KIBXJ1WD6SWJW0IFBTHGCFU1SKOKK", "3NPI0JQDAP3D7F26OKKO637GUJPPT8", "3SITXWYCNW7IK2AGAP3K0MNXQ3DBXR"],
        "reverse": ["39L1G8WVWRP5R6LAO337NULKX9931E"],
        "biased": [],
        "fake": ["9929"]
    }
    for result in results:
        # Skip results with IDs in the exclusion list for the current task
        if result.get('sample_id') in exclude_ids_by_task.get(task, []):
            continue
        parsed_response = result.get('parsed_response', {})
        harmful_response = parsed_response.get('harmful_response', 'unknown')
        harmful_responses.append(harmful_response)
    
    harmful_response_counts = Counter(harmful_responses)
    return harmful_response_counts


def print_statistics(results, task):
    """Print statistics"""
    print(f"\n{'='*60}")
    print(f"Overall statistics - {len(results)} samples")
    print(f"{'='*60}")
    
    # 1. Harmful Request distribution
    harmful_request_counts = analyze_harmful_requests(results)
    print("\nHarmful Request distribution:")
    print("-" * 30)
    total_samples = sum(harmful_request_counts.values())
    for request_type, count in harmful_request_counts.most_common():
        percentage = (count / total_samples) * 100
        print(f"  {count}/{total_samples}=({percentage:5.1f}%)")
        print(f"  {request_type:10}: {count:4} ({percentage:5.1f}%)")
    
    # 2. Harmful Response distribution
    harmful_response_counts = analyze_harmful_responses(results, task)
    print("\nHarmful Response distribution:")
    print("-" * 30)
    total_samples = sum(harmful_response_counts.values())
    for response_type, count in harmful_response_counts.most_common():
        percentage = (count / total_samples) * 100
        print(f"  {count}/{total_samples}=({percentage:5.1f}%)")
        print(f"  {response_type:10}: {count:4} ({percentage:5.1f}%)")



In [ ]:
for task in ["vague", "reverse", "biased", "fake"]:    
    results_dir = f'./results_{task}_wild_resp'
    if not os.path.exists(results_dir):
        print(f"Error: directory {results_dir} does not exist")
    # Load results
    print(f"Loading results from {results_dir}...")
    results = load_results_from_directory(results_dir)
    if not results:
        print("Error: no valid result files found")
    # Print statistics
    print_statistics(results, task)

Loading results from ./results_vague_wild_resp...
Found 2500 result files

Overall statistics - 2500 samples

Harmful Request distribution:
------------------------------
  1932/2500=( 77.3%)
  no        : 1932 ( 77.3%)
  568/2500=( 22.7%)
  yes       :  568 ( 22.7%)

Harmful Response distribution:
------------------------------
  1964/2496=( 78.7%)
  no        : 1964 ( 78.7%)
  532/2496=( 21.3%)
  yes       :  532 ( 21.3%)
Loading results from ./results_reverse_wild_resp...
Found 2500 result files

Overall statistics - 2500 samples

Harmful Request distribution:
------------------------------
  2420/2500=( 96.8%)
  no        : 2420 ( 96.8%)
  80/2500=(  3.2%)
  yes       :   80 (  3.2%)

Harmful Response distribution:
------------------------------
  2105/2499=( 84.2%)
  no        : 2105 ( 84.2%)
  394/2499=( 15.8%)
  yes       :  394 ( 15.8%)
Loading results from ./results_biased_wild_resp...
Found 2800 result files

Overall statistics - 2800 samples

Harmful Request distribution:
--